# Детекция, трекинг и подсчет автомобилей

Ноутбук решает задание по этапам: находит автомобили YOLOv8, ведет треки через BoT-SORT, считает пересечение заданной линии в одном направлении, сохраняет размеченное видео и CSV-отчет. Дополнительно добавлена OCR-логика для распознавания номеров: специализированная YOLOv8-модель номерных знаков автоматически скачивается с Hugging Face и используется перед EasyOCR.

## 0. Установка зависимостей

Эта ячейка устанавливает зависимости из `requirements.txt`, чтобы ноутбук можно было запустить на другом компьютере. Если CUDA доступна, PyTorch/Ultralytics будут использовать GPU автоматически.

In [ ]:
import sys
import subprocess
from pathlib import Path

requirements_path = Path('requirements.txt')
if not requirements_path.exists():
    raise FileNotFoundError('Файл requirements.txt не найден рядом с ноутбуком')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)])

## 1. Импорт библиотек и выбор устройства

Здесь подключаются основные библиотеки. Устройство выбирается автоматически: `cuda:0`, если доступна видеокарта NVIDIA, иначе `cpu`.

In [ ]:
from collections import defaultdict
from pathlib import Path
import csv
import re
import time

import cv2
import numpy as np
import pandas as pd
import torch
from IPython.display import Video, display
from tqdm.auto import tqdm
from ultralytics import YOLO

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Устройство для инференса: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Настройки эксперимента

Можно выбрать конкретное видео или оставить `VIDEO_PATH = None`, тогда будет автоматически взято первое найденное `.mp4` из папки `drivingDataset`. Линия подсчета задается двумя точками; если оставить `COUNTING_LINE = None`, будет построена вертикальная линия на 65% ширины кадра. Для выбранного демонстрационного видео это дает пересечения уже на коротком фрагменте.

In [ ]:
DATASET_DIR = Path('drivingDataset')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Можно заменить на конкретный файл или поставить None для автоматического выбора первого .mp4.
VIDEO_PATH = Path('drivingDataset/normalDay/nD_5.mp4')

# Модель YOLOv8. Для скорости можно оставить yolov8n.pt, для качества попробовать yolov8s.pt или yolov8m.pt.
YOLO_MODEL = 'yolov8n.pt'
TRACKER_CONFIG = 'botsort.yaml'

# В COCO class_id=2 соответствует классу car. При необходимости можно добавить bus=5, truck=7.
CLASSES_TO_TRACK = [2]
CONF_THRESHOLD = 0.35
IOU_THRESHOLD = 0.5

# None обрабатывает все видео. Для быстрой проверки можно поставить 300-600 кадров.
MAX_FRAMES = 600

# Линия подсчета: ((x1, y1), (x2, y2)). None построит вертикальную линию на 65% ширины кадра.
COUNTING_LINE = None

# Для вертикальной линии сверху вниз: positive_to_negative означает движение слева направо.
COUNT_DIRECTION = 'positive_to_negative'  # варианты: positive_to_negative, negative_to_positive
LINE_MARGIN_PX = 2

# Дополнительное задание: детекция и OCR номеров.
ENABLE_PLATE_OCR = True
AUTO_DOWNLOAD_PLATE_MODEL = True
PLATE_MODEL_REPO_ID = 'Koushim/yolov8-license-plate-detection'
PLATE_MODEL_FILENAME = 'best.pt'
PLATE_DETECTOR_MODEL = Path('models/license_plate_detector.pt')
OCR_EVERY_N_FRAMES = 15
MIN_PLATE_TEXT_LEN = 4

## 3. Поиск видео и чтение параметров

Эта ячейка находит входное видео, проверяет, что оно открывается, считывает размер кадра и FPS, а также автоматически задает линию подсчета, если она не была задана вручную.

In [ ]:
def find_video(dataset_dir: Path) -> Path:
    videos = sorted(dataset_dir.rglob('*.mp4'))
    if not videos:
        raise FileNotFoundError(f'В папке {dataset_dir.resolve()} не найдено .mp4 видео')
    return videos[0]

input_video = Path(VIDEO_PATH) if VIDEO_PATH is not None else find_video(DATASET_DIR)
if not input_video.exists():
    raise FileNotFoundError(f'Видео не найдено: {input_video}')

cap = cv2.VideoCapture(str(input_video))
if not cap.isOpened():
    raise RuntimeError(f'Не удалось открыть видео: {input_video}')

fps = cap.get(cv2.CAP_PROP_FPS) or 25
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

if COUNTING_LINE is None:
    x = int(frame_width * 0.65)
    counting_line = ((x, int(frame_height * 0.15)), (x, int(frame_height * 0.95)))
else:
    counting_line = COUNTING_LINE

output_video_path = OUTPUT_DIR / f'{input_video.stem}_tracked_counted.mp4'
output_csv_path = OUTPUT_DIR / f'{input_video.stem}_report.csv'

print(f'Видео: {input_video}')
print(f'Размер: {frame_width}x{frame_height}, FPS: {fps:.2f}, кадров: {frame_count}')
print(f'Линия подсчета: {counting_line}')
print(f'Выходное видео: {output_video_path}')
print(f'CSV-отчет: {output_csv_path}')

## 4. Геометрия пересечения линии

Для каждого объекта используется центр bounding box. Пересечение считается по смене стороны относительно линии и только в выбранном направлении. Один и тот же `track_id` после первого пересечения повторно не учитывается.

In [ ]:
def signed_side(point, line):
    (x1, y1), (x2, y2) = line
    px, py = point
    return (x2 - x1) * (py - y1) - (y2 - y1) * (px - x1)

def crossed_in_direction(prev_point, curr_point, line, direction='positive_to_negative', margin=2):
    prev_side = signed_side(prev_point, line)
    curr_side = signed_side(curr_point, line)

    if direction == 'positive_to_negative':
        return prev_side > margin and curr_side <= -margin
    if direction == 'negative_to_positive':
        return prev_side < -margin and curr_side >= margin
    raise ValueError('COUNT_DIRECTION должен быть positive_to_negative или negative_to_positive')

def box_center(xyxy):
    x1, y1, x2, y2 = xyxy
    return (int((x1 + x2) / 2), int((y1 + y2) / 2))

def normalize_plate_text(text):
    text = text.upper()
    text = re.sub(r'[^A-ZА-Я0-9]', '', text)
    return text

def draw_text_with_bg(frame, text, origin, color=(255, 255, 255), bg_color=(0, 0, 0), scale=0.6, thickness=2):
    x, y = origin
    (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thickness)
    cv2.rectangle(frame, (x, y - th - baseline - 4), (x + tw + 4, y + 4), bg_color, -1)
    cv2.putText(frame, text, (x + 2, y - baseline), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)

## 5. Детекция и распознавание номеров

Эта часть автоматически скачивает специализированную YOLOv8-модель детекции номерных знаков `Koushim/yolov8-license-plate-detection` в `models/license_plate_detector.pt`. Затем модель ищет номер внутри bounding box автомобиля, а EasyOCR распознает текст только на найденной области. Если интернет недоступен, основной трекинг автомобилей продолжит работать, но OCR перейдет на запасную эвристику по нижней части автомобиля.

In [ ]:
def ensure_plate_detector_model(model_path, repo_id, filename='best.pt', auto_download=True):
    model_path = Path(model_path) if model_path else None
    if model_path is None:
        return None
    if model_path.exists():
        return model_path
    if not auto_download:
        print(f'Модель номеров не найдена: {model_path}')
        return None

    try:
        from huggingface_hub import hf_hub_download
        model_path.parent.mkdir(parents=True, exist_ok=True)
        downloaded_path = hf_hub_download(repo_id=repo_id, filename=filename)
        import shutil
        shutil.copy2(downloaded_path, model_path)
        print(f'Модель номеров скачана: {model_path}')
        return model_path
    except Exception as exc:
        print(f'Не удалось скачать модель номеров с Hugging Face: {exc}')
        print(f'Можно скачать вручную: https://huggingface.co/{repo_id}/resolve/main/{filename}')
        print(f'И сохранить файл как: {model_path}')
        return None

def build_plate_tools(enable_ocr=True, plate_model_path=None):
    reader = None
    plate_model = None

    if enable_ocr:
        try:
            import easyocr
            reader = easyocr.Reader(['en', 'ru'], gpu=torch.cuda.is_available())
            print('EasyOCR загружен')
        except Exception as exc:
            print(f'OCR отключен: EasyOCR не удалось загрузить ({exc})')

    if plate_model_path:
        plate_model_path = Path(plate_model_path)
        if plate_model_path.exists():
            plate_model = YOLO(str(plate_model_path))
            print(f'Детектор номеров загружен: {plate_model_path}')
        else:
            print(f'Модель номеров не найдена: {plate_model_path}. Будет использована эвристика ROI.')

    return reader, plate_model

def recognize_plate(frame, car_box, reader=None, plate_model=None):
    if reader is None:
        return ''

    x1, y1, x2, y2 = [int(v) for v in car_box]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(frame.shape[1] - 1, x2), min(frame.shape[0] - 1, y2)
    car_crop = frame[y1:y2, x1:x2]
    if car_crop.size == 0:
        return ''

    plate_crops = []
    if plate_model is not None:
        plate_results = plate_model.predict(car_crop, conf=0.25, verbose=False, device=DEVICE)
        for result in plate_results:
            if result.boxes is None:
                continue
            for plate_box in result.boxes.xyxy.cpu().numpy():
                px1, py1, px2, py2 = [int(v) for v in plate_box]
                crop = car_crop[max(0, py1):max(0, py2), max(0, px1):max(0, px2)]
                if crop.size > 0:
                    plate_crops.append(crop)

    if not plate_crops:
        h, w = car_crop.shape[:2]
        plate_crops.append(car_crop[int(h * 0.45):h, int(w * 0.10):int(w * 0.90)])

    candidates = []
    for crop in plate_crops:
        if crop.size == 0:
            continue
        ocr_results = reader.readtext(crop, detail=1, paragraph=False)
        for _, text, conf in ocr_results:
            normalized = normalize_plate_text(text)
            if len(normalized) >= MIN_PLATE_TEXT_LEN:
                candidates.append((normalized, float(conf)))

    if not candidates:
        return ''
    candidates.sort(key=lambda item: item[1], reverse=True)
    return candidates[0][0]

## 6. Основная обработка видео

В этой ячейке запускается YOLOv8 с BoT-SORT. На каждом кадре рисуются bounding boxes, `track_id`, класс, центр объекта, линия подсчета и общий счетчик пересечений. CSV фиксирует первое пересечение каждого объекта и, если OCR сработал, распознанный номер.

In [ ]:
model = YOLO(YOLO_MODEL)
resolved_plate_model_path = ensure_plate_detector_model(
    PLATE_DETECTOR_MODEL,
    PLATE_MODEL_REPO_ID,
    PLATE_MODEL_FILENAME,
    AUTO_DOWNLOAD_PLATE_MODEL and ENABLE_PLATE_OCR,
)
ocr_reader, plate_detector = build_plate_tools(ENABLE_PLATE_OCR, resolved_plate_model_path)

cap = cv2.VideoCapture(str(input_video))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(str(output_video_path), fourcc, fps, (frame_width, frame_height))

last_centers = {}
counted_ids = set()
track_plate_text = defaultdict(str)
events = []
total_crossings = 0
processed_frames = 0
total_frames_to_process = min(frame_count, MAX_FRAMES) if MAX_FRAMES else frame_count
start_time = time.time()

progress = tqdm(total=total_frames_to_process, desc='Обработка кадров')

while True:
    ok, frame = cap.read()
    if not ok:
        break
    if MAX_FRAMES and processed_frames >= MAX_FRAMES:
        break

    frame_index = processed_frames
    timestamp_sec = frame_index / fps

    results = model.track(
        frame,
        persist=True,
        tracker=TRACKER_CONFIG,
        classes=CLASSES_TO_TRACK,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )

    cv2.line(frame, counting_line[0], counting_line[1], (0, 255, 255), 3)

    for result in results:
        boxes = result.boxes
        if boxes is None or boxes.id is None:
            continue

        xyxy_list = boxes.xyxy.cpu().numpy()
        track_ids = boxes.id.int().cpu().tolist()
        class_ids = boxes.cls.int().cpu().tolist()
        confidences = boxes.conf.cpu().tolist()

        for xyxy, track_id, class_id, confidence in zip(xyxy_list, track_ids, class_ids, confidences):
            x1, y1, x2, y2 = [int(v) for v in xyxy]
            center = box_center((x1, y1, x2, y2))
            prev_center = last_centers.get(track_id)

            crossed = False
            if prev_center is not None and track_id not in counted_ids:
                crossed = crossed_in_direction(prev_center, center, counting_line, COUNT_DIRECTION, LINE_MARGIN_PX)
                if crossed:
                    counted_ids.add(track_id)
                    total_crossings += 1

            if ENABLE_PLATE_OCR and (not track_plate_text[track_id]) and frame_index % OCR_EVERY_N_FRAMES == 0:
                plate_text = recognize_plate(frame, (x1, y1, x2, y2), ocr_reader, plate_detector)
                if plate_text:
                    track_plate_text[track_id] = plate_text

            if crossed:
                events.append({
                    'track_id': track_id,
                    'class_id': class_id,
                    'class_name': model.names.get(class_id, str(class_id)),
                    'frame': frame_index,
                    'time_sec': round(timestamp_sec, 3),
                    'center_x': center[0],
                    'center_y': center[1],
                    'plate_text': track_plate_text[track_id],
                })

            last_centers[track_id] = center

            was_counted = track_id in counted_ids
            box_color = (0, 180, 0) if was_counted else (255, 80, 0)
            center_color = (0, 255, 0) if was_counted else (0, 0, 255)
            label = f'ID {track_id} {model.names.get(class_id, class_id)} {confidence:.2f}'
            if was_counted:
                label += ' counted'
            if track_plate_text[track_id]:
                label += f' plate:{track_plate_text[track_id]}'

            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
            cv2.circle(frame, center, 5, center_color, -1)
            draw_text_with_bg(frame, label, (x1, max(25, y1 - 6)), color=(255, 255, 255), bg_color=box_color, scale=0.55)

    draw_text_with_bg(frame, f'Count: {total_crossings}', (20, 45), color=(255, 255, 255), bg_color=(30, 30, 30), scale=1.1, thickness=2)
    draw_text_with_bg(frame, f'Direction: {COUNT_DIRECTION}', (20, 85), color=(255, 255, 255), bg_color=(30, 30, 30), scale=0.7, thickness=2)

    writer.write(frame)
    processed_frames += 1
    progress.update(1)

progress.close()
cap.release()
writer.release()

events_df = pd.DataFrame(events)
events_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

elapsed = time.time() - start_time
print(f'Готово. Обработано кадров: {processed_frames}')
print(f'Найдено пересечений: {total_crossings}')
print(f'Время обработки: {elapsed:.1f} сек')
print(f'Видео сохранено: {output_video_path}')
print(f'CSV сохранен: {output_csv_path}')
display(events_df.head(20))

## 7. Просмотр результата

После обработки можно прямо в ноутбуке открыть получившееся видео и проверить, что на кадрах есть bounding boxes, ID, центры объектов, линия и счетчик пересечений.

In [ ]:
if output_video_path.exists():
    display(Video(str(output_video_path), embed=True, width=900))
else:
    print('Выходное видео пока не создано')

if output_csv_path.exists():
    report_df = pd.read_csv(output_csv_path)
    display(report_df)

## 8. Что изменить для полного эксперимента

Для обработки полного видео поставьте `MAX_FRAMES = None`. Если автомобили едут не слева направо, поменяйте `COUNT_DIRECTION` или вручную задайте `COUNTING_LINE`. Модель детекции номеров скачивается автоматически в `models/license_plate_detector.pt`; если интернет недоступен, скачайте `best.pt` с Hugging Face вручную и сохраните его по этому пути.